# RAG Pipeline — Retrieval Augmented Generation
**Author:** Juan Esteban Agudelo Ortiz  
**Email:** juan.es.agor@gmail.com

---

This notebook implements the complete RAG pipeline by connecting the
vector index built in the embeddings notebook with a local language
model (Qwen2.5-3B-Instruct) to generate structured flash cards from
retrieved context.

A RAG pipeline has two stages:

1. **Retrieval:** given a query, find the $k$ most semantically similar
   chunks from the vector index using cosine similarity.
2. **Augmented generation:** construct a prompt that includes the
   retrieved chunks as context, and pass it to the language model to
   generate a structured response.

Without retrieval, the model generates from its parametric knowledge
alone, which may be incomplete or outdated. With retrieval, the model
grounds its response in the provided context, reducing hallucinations
and improving factual accuracy.

### Limitations
1. Response quality is bounded by retrieval quality; irrelevant chunks
   produce irrelevant flash cards.
2. Qwen2.5-3B has a context window of 2048 tokens; prompts exceeding
   this limit are truncated silently.
3. Structured output compliance depends on prompt constraints; schema
   violations are possible and must be caught by output validation.
4. Generation on CPU is slow; expect 1-3 minutes per flash card on
   hardware without GPU.

## 0. Install dependencies

In [11]:
# Run only once
# !pip install llama-index-core llama-index-embeddings-huggingface chromadb llama-index-vector-stores-chroma llama-cpp-python

## 1. Imports and configuration

In [12]:
import re
import json
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum

import chromadb
from llama_index.core import VectorStoreIndex, StorageContext, Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_cpp import Llama

# --- Directory setup ---
BASE_DIR    = Path("..")
UPLOADS_DIR = BASE_DIR / "data" / "uploads"
OUT_DIR     = BASE_DIR / "outputs"
INDEX_DIR   = BASE_DIR / "data" / "index"
MODELS_DIR  = BASE_DIR / "data" / "models"

UPLOADS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# --- Constants ---
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODEL_FILENAME       = "qwen2.5-3b-instruct-q4_k_m.gguf"
MODEL_PATH           = MODELS_DIR / MODEL_FILENAME

# --- Reproduce IngestedDocument from notebook 1 ---
class InputType(Enum):
    PDF           = "pdf"
    HANDWRITTEN   = "handwritten_image"
    REFERENCE_IMG = "reference_image"
    PLAIN_TEXT    = "plain_text"

@dataclass
class IngestedDocument:
    input_type       : InputType
    text             : str
    reference_images : list = field(default_factory=list)
    source_path      : Optional[Path] = None

print(f"Uploads dir : {UPLOADS_DIR.resolve()}")
print(f"Models dir  : {MODELS_DIR.resolve()}")
print(f"Index dir   : {INDEX_DIR.resolve()}")
print("Configuration ready ✓")

Uploads dir : /home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/data/uploads
Models dir  : /home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/data/models
Index dir   : /home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/data/index
Configuration ready ✓


## 2. Language model setup

Qwen2.5-3B-Instruct is loaded in GGUF format with 4-bit quantization
(Q4_K_M). Quantization reduces the precision of model weights from
32-bit floats to 4-bit integers, reducing RAM usage from ~6GB to ~2.5GB
at the cost of a small degradation in output quality.

The model is loaded with `llama-cpp-python`, which runs inference
entirely on CPU using optimized C++ kernels. Key parameters:

- `n_ctx`: context window size in tokens. Set to 2048 for Qwen2.5-3B.
- `n_threads`: number of CPU threads for inference. Set to the number
  of physical cores available.
- `verbose`: suppresses llama.cpp internal logs when set to `False`.

In [13]:
# Download model if not present — runs only once (~2GB)
from huggingface_hub import hf_hub_download
import os

MODEL_REPO     = "Qwen/Qwen2.5-3B-Instruct-GGUF"
MODEL_FILENAME = "qwen2.5-3b-instruct-q4_k_m.gguf"
MODEL_PATH     = MODELS_DIR / MODEL_FILENAME

# Note: this download does not require authentication since the model is public.
# If you have a HuggingFace token and want faster downloads, set the environment
# variable HF_TOKEN before running this cell. Never hardcode your token in the notebook.
if not MODEL_PATH.exists():
    print(f"Downloading {MODEL_FILENAME} (~2GB)...")
    hf_hub_download(
        repo_id   = MODEL_REPO,
        filename  = MODEL_FILENAME,
        local_dir = str(MODELS_DIR),
    )
    print("Download complete ✓")
else:
    print(f"Model already present: {MODEL_PATH}")

Model already present: ../data/models/qwen2.5-3b-instruct-q4_k_m.gguf


In [14]:
def load_language_model(model_path: Path, n_ctx: int = 2048) -> Llama:
    """
    Load Qwen2.5-3B-Instruct in GGUF format using llama-cpp-python.

    Parameters
    ----------
    model_path : Path
        Path to the GGUF model file.
    n_ctx : int
        Context window size in tokens. Default 2048 for Qwen2.5-3B.

    Returns
    -------
    Llama
        Loaded language model ready for inference.
    """
    import os
    n_threads = os.cpu_count()

    print(f"Loading model from : {model_path}")
    print(f"Context window     : {n_ctx} tokens")
    print(f"CPU threads        : {n_threads}")

    llm = Llama(
        model_path = str(model_path),
        n_ctx      = n_ctx,
        n_threads  = n_threads,
        verbose    = False,
    )

    print("Language model ready ✓")
    return llm


# Initialize model — takes 30-60 seconds on first load
llm = load_language_model(MODEL_PATH)

Loading model from : ../data/models/qwen2.5-3b-instruct-q4_k_m.gguf
Context window     : 2048 tokens
CPU threads        : 8


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Language model ready ✓


## 3. Prompt templates

A prompt template defines the structure of the input sent to the language
model. For RAG, the prompt has three components:

1. **System message:** defines the model's role and output format
2. **Context:** the retrieved chunks concatenated as reference material
3. **User query:** the topic the user wants a flash card about

The model receives these three components formatted as a chat conversation
using the Qwen2.5 chat template:

```text
<|im_start|>system
{system_message}
<|im_end|>
<|im_start|>user
Context:
{context}

Question: {query}
<|im_end|>
<|im_start|>assistant
```

The output schema defines the fields the model must produce. Two schemas
are used depending on the output mode selected by the user:

- **Individual flash card:** one concept with definition, examples, and
  suggested image
- **Consolidated summary:** multiple concepts with brief descriptions

In [15]:
SYSTEM_PROMPT_FLASHCARD = """You are an expert study assistant that creates structured flash cards for students.

Given a context extracted from a textbook and a topic query, generate a flash card in JSON format.
The JSON must follow this exact schema with no additional fields:

{
    "concept": "name of the concept",
    "definition": "clear and concise definition in 2-3 sentences",
    "key_points": ["point 1", "point 2", "point 3"],
    "examples": ["example 1", "example 2"],
    "suggested_image": "brief description of an image that would illustrate this concept"
}

Rules:
- Respond ONLY with the JSON object, no preamble, no explanation, no markdown backticks
- Base your response strictly on the provided context
- If the context does not contain enough information, still follow the schema but indicate the limitation in the definition field
- All fields are required"""


SYSTEM_PROMPT_SUMMARY = """You are an expert study assistant that creates structured summaries for students.

Given a context extracted from a textbook and a topic query, generate a consolidated summary in JSON format.
The JSON must follow this exact schema with no additional fields:

{
    "topic": "main topic name",
    "overview": "2-3 sentence overview of the topic",
    "concepts": [
        {
            "name": "concept name",
            "description": "brief description in 1-2 sentences"
        }
    ]
}

Rules:
- Respond ONLY with the JSON object, no preamble, no explanation, no markdown backticks
- Base your response strictly on the provided context
- Include between 3 and 8 concepts
- All fields are required"""


VALID_MODES = {"flashcard", "summary"}

def build_prompt(
    query: str,
    context_chunks: list[dict],
    mode: str = "flashcard",
) -> list[dict]:
    
    """
    Build a chat prompt for the language model.

    Parameters
    ----------
    query : str
        The topic or question from the user.
    context_chunks : list[dict]
        Retrieved chunks from the vector store.
    mode : str
        Output mode: 'flashcard' for individual cards, 'summary' for
        consolidated summaries. Default 'flashcard'.

    Returns
    -------
    list[dict]
        Chat messages in the format expected by llama-cpp-python.
    """
    if mode not in VALID_MODES:
        raise ValueError(f"Invalid mode '{mode}'. Must be one of {VALID_MODES}")
    
    system_prompt = (
        SYSTEM_PROMPT_FLASHCARD if mode == "flashcard"
        else SYSTEM_PROMPT_SUMMARY
    )

    context = "\n\n".join(
        f"[Chunk {i+1}]\n{chunk['text']}"
        for i, chunk in enumerate(context_chunks)
    )

    return [
        {"role": "system",    "content": system_prompt},
        {"role": "user",      "content": f"Context:\n{context}\n\nTopic: {query}"},
    ]


print("Prompt templates defined ✓")

Prompt templates defined ✓


## 4. RAG query pipeline

The RAG pipeline connects the vector store from the embeddings notebook
with the language model. For each user query the pipeline executes
four steps in sequence:

1. **Embed the query** using the same embedding model used for indexing
2. **Retrieve** the $k$ most similar chunks from the vector store
3. **Build the prompt** by injecting the retrieved chunks as context
4. **Generate** the structured response using the language model

The four steps form a directed pipeline where the output of each step
is the input of the next. A failure in any step propagates downstream,
so each step is wrapped in explicit error handling.

In [16]:
def rag_query(
    query: str,
    index: VectorStoreIndex,
    llm: Llama,
    embed_model: HuggingFaceEmbedding,
    mode: str = "flashcard",
    k: int = 3,
    max_tokens: int = 512,
) -> dict:
    """
    Run a full RAG query: retrieve context, build prompt, generate response.

    Parameters
    ----------
    query : str
        Topic or question from the user.
    index : VectorStoreIndex
        Vector store index built from the document chunks.
    llm : Llama
        Loaded language model.
    embed_model : HuggingFaceEmbedding
        Embedding model used for indexing. Must match the index.
    mode : str
        Output mode: 'flashcard' or 'summary'. Default 'flashcard'.
    k : int
        Number of chunks to retrieve. Default 3.
    max_tokens : int
        Maximum tokens to generate. Default 512.

    Returns
    -------
    dict
        Parsed JSON response from the language model.

    Raises
    ------
    ValueError
        If retrieval returns no chunks or generation fails.
    """
    # Step 1 & 2: embed query and retrieve chunks
    retriever = index.as_retriever(
        similarity_top_k = k,
        embed_model      = embed_model,
    )
    chunks = retriever.retrieve(query)

    if not chunks:
        raise ValueError(f"No chunks retrieved for query: '{query}'")

    chunk_dicts = [
        {"text": node.text, "score": node.score}
        for node in chunks
    ]

    # Step 3: build prompt
    messages = build_prompt(query, chunk_dicts, mode=mode)

    # Step 4: generate response
    response = llm.create_chat_completion(
        messages   = messages,
        max_tokens = max_tokens,
        temperature = 0.1,
    )

    raw_text = response["choices"][0]["message"]["content"].strip()

    # Parse JSON response
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError as e:
        raise ValueError(f"Model did not return valid JSON: {e}\nRaw output: {raw_text}")


print("rag_query defined ✓")

rag_query defined ✓


## 5. Structured output for flash cards

The raw output of `rag_query` is a Python dict parsed from JSON. Before
passing it to the PDF generator, we validate that all required fields
are present and have the correct types.

Two dataclasses represent the two output modes:

- `FlashCard`: single concept with definition, key points, examples,
  and suggested image
- `ConsolidatedSummary`: multiple concepts with brief descriptions

Validation catches schema violations before they propagate to the PDF
generation stage, following the fail fast principle established in the
ingestion pipeline.

In [17]:
@dataclass
class FlashCard:
    """
    Structured representation of a single concept flash card.

    Attributes
    ----------
    concept : str
        Name of the concept.
    definition : str
        Clear and concise definition in 2-3 sentences.
    key_points : list[str]
        List of key points about the concept.
    examples : list[str]
        Concrete examples of the concept.
    suggested_image : str
        Brief description of an illustrative image.
    """
    concept         : str
    definition      : str
    key_points      : list
    examples        : list
    suggested_image : str


@dataclass
class ConsolidatedSummary:
    """
    Structured representation of a consolidated summary.

    Attributes
    ----------
    topic : str
        Main topic name.
    overview : str
        2-3 sentence overview of the topic.
    concepts : list[dict]
        List of concepts, each with 'name' and 'description' fields.
    """
    topic    : str
    overview : str
    concepts : list


def parse_response(raw: dict, mode: str) -> FlashCard | ConsolidatedSummary:
    """
    Parse and validate the raw JSON response from the language model.

    Parameters
    ----------
    raw : dict
        Parsed JSON dict from rag_query.
    mode : str
        Output mode: 'flashcard' or 'summary'.

    Returns
    -------
    FlashCard or ConsolidatedSummary
        Validated structured output.

    Raises
    ------
    ValueError
        If required fields are missing from the response.
    """
    if mode not in VALID_MODES:
        raise ValueError(f"Invalid mode '{mode}'. Must be one of {VALID_MODES}")

    try:
        if mode == "flashcard":
            return FlashCard(
                concept         = raw["concept"],
                definition      = raw["definition"],
                key_points      = raw["key_points"],
                examples        = raw["examples"],
                suggested_image = raw["suggested_image"],
            )
        else:
            return ConsolidatedSummary(
                topic    = raw["topic"],
                overview = raw["overview"],
                concepts = raw["concepts"],
            )
    except KeyError as e:
        raise ValueError(f"Schema violation — missing field: {e}\nRaw response: {raw}")


print("FlashCard and ConsolidatedSummary defined ✓")

FlashCard and ConsolidatedSummary defined ✓


## 6. Output validation

Output validation adds one retry when the model produces a schema
violation. On the first attempt the standard prompt is used. If parsing
fails, a second attempt is made with a stricter prompt that explicitly
shows an example of the expected JSON structure.

A second consecutive schema violation raises a `ValueError` and the
user is notified to rephrase their query or check that the uploaded
document contains relevant information about the requested topic.

In [18]:
FLASHCARD_EXAMPLE = """{
    "concept": "Amide",
    "definition": "An amide is a compound derived from a carboxylic acid where the hydroxyl group is replaced by an amino group. Amides are found in proteins and many biological molecules.",
    "key_points": [
        "Derived from carboxylic acids",
        "Contains a carbonyl group bonded to nitrogen",
        "Found in proteins as peptide bonds"
    ],
    "examples": [
        "Acetamide (CH3CONH2)",
        "Nylon (synthetic polyamide)"
    ],
    "suggested_image": "Structural formula of an amide showing the carbonyl group bonded to nitrogen"
}"""

SUMMARY_EXAMPLE = """{
    "topic": "Carboxylic Acid Derivatives",
    "overview": "Carboxylic acid derivatives are compounds that can be hydrolyzed to give carboxylic acids. They include esters, amides, anhydrides, and acid chlorides.",
    "concepts": [
        {"name": "Ester", "description": "Formed by reaction of carboxylic acid with alcohol"},
        {"name": "Amide", "description": "Formed by reaction of carboxylic acid with amine"}
    ]
}"""


def rag_query_with_validation(
    query: str,
    index: VectorStoreIndex,
    llm: Llama,
    embed_model: HuggingFaceEmbedding,
    mode: str = "flashcard",
    k: int = 3,
    max_tokens: int = 512,
) -> FlashCard | ConsolidatedSummary:

    example = FLASHCARD_EXAMPLE if mode == "flashcard" else SUMMARY_EXAMPLE
    base_prompt = (
        SYSTEM_PROMPT_FLASHCARD if mode == "flashcard"
        else SYSTEM_PROMPT_SUMMARY
    )

    prompts_to_try = [
        base_prompt,
        base_prompt + f"\n\nExample of expected output:\n{example}",
    ]

    last_error = None
    for attempt, system_prompt in enumerate(prompts_to_try):
        try:
            retriever = index.as_retriever(
                similarity_top_k = k,
                embed_model      = embed_model,
            )
            chunks = retriever.retrieve(query)

            if not chunks:
                raise ValueError(f"No chunks retrieved for query: '{query}'")

            chunk_dicts = [
                {"text": node.text, "score": node.score}
                for node in chunks
            ]

            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": f"Context:\n{chr(10).join(c['text'] for c in chunk_dicts)}\n\nTopic: {query}"},
            ]

            response = llm.create_chat_completion(
                messages    = messages,
                max_tokens  = max_tokens,
                temperature = 0.1,
            )

            raw_text = response["choices"][0]["message"]["content"].strip()
            raw      = json.loads(raw_text)
            return parse_response(raw, mode)

        except (ValueError, json.JSONDecodeError) as e:
            last_error = e
            if attempt == 0:
                print(f"Schema violation on attempt 1: {e}")
                print("Retrying with example-augmented prompt...")

    raise ValueError(
        f"Schema violation on both attempts. "
        f"Try rephrasing your query or verify that the document "
        f"contains information about '{query}'.\n"
        f"Last error: {last_error}"
    )


print("rag_query_with_validation defined ✓")

rag_query_with_validation defined ✓


## 7. Demo — generating a flash card from OpenStax chapter 21

This demo runs the full RAG pipeline on the carboxylic acid derivatives
chapter from OpenStax Organic Chemistry, generating both a flash card
and a consolidated summary for chemistry topics relevant to biology students.

### Getting the PDF
Download the full book from:
```text
https://openstax.org/details/books/organic-chemistry
```
Then extract chapter 21 (pages 741 to 792) using PyMuPDF:

```python
import fitz
doc = fitz.open("openstax_organic_chemistry.pdf")
sub = fitz.open()
sub.insert_pdf(doc, from_page=740, to_page=791)
sub.save("data/uploads/openstax_ch21_carboxylic_acid_derivatives.pdf")
```

Note: PyMuPDF uses zero-based page indexing, so page 741 corresponds to index 740.

In [19]:
import fitz
import re
from llama_index.core.node_parser import SentenceSplitter

# --- Reproduce pipeline from previous notebooks ---
def extract_text_from_pdf(pdf_path: Path, min_block_chars: int = 20) -> str:
    """
    Extract ordered text from a PDF using block-based extraction.
    Reproduced from the ingestion pipeline notebook.
    """
    doc = fitz.open(pdf_path)
    all_text = []

    for page_num, page in enumerate(doc):
        blocks = page.get_text("blocks")
        blocks_sorted = sorted(blocks, key=lambda b: (b[1], b[0]))

        page_text = []
        for block in blocks_sorted:
            text = block[4].strip()
            if len(text) < min_block_chars:
                continue
            text = re.sub(r"\s+", " ", text)
            page_text.append(text)

        if page_text:
            all_text.append(f"--- Page {page_num + 1} ---\n" + "\n\n".join(page_text))

    doc.close()
    return "\n\n".join(all_text)


def fixed_size_chunking(doc: IngestedDocument, chunk_size: int = 512, chunk_overlap: int = 64) -> list[dict]:
    """
    Split document text into fixed-size chunks with overlap.
    Reproduced from the chunking pipeline notebook.
    """
    splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    llama_doc = Document(text=doc.text)
    nodes = splitter.get_nodes_from_documents([llama_doc])
    return [{"text": node.text, "index": i, "strategy": "fixed_size"} for i, node in enumerate(nodes)]


METADATA_FILE = "index_metadata.json"

def create_vector_store(collection_name: str, persist_dir: Path):
    chroma_client = chromadb.PersistentClient(path=str(persist_dir))
    collection    = chroma_client.get_or_create_collection(collection_name)
    vector_store  = ChromaVectorStore(chroma_collection=collection)
    return collection, vector_store

def build_index(chunks, vector_store, embed_model):
    documents = [
        Document(text=chunk["text"], metadata={"chunk_index": chunk["index"], "strategy": chunk["strategy"]})
        for chunk in chunks
    ]
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    return VectorStoreIndex.from_documents(documents, storage_context=storage_context, embed_model=embed_model, show_progress=True)

def load_and_verify_index(persist_dir, collection_name, current_model_name, embed_model, chunks):
    metadata_path = persist_dir / METADATA_FILE
    if metadata_path.exists():
        stored = json.loads(metadata_path.read_text(encoding="utf-8"))
        if stored.get("embedding_model") != current_model_name:
            import shutil
            shutil.rmtree(persist_dir)
            persist_dir.mkdir(parents=True, exist_ok=True)
        else:
            _, vector_store = create_vector_store(collection_name, persist_dir)
            return VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)
    _, vector_store = create_vector_store(collection_name, persist_dir)
    index = build_index(chunks, vector_store, embed_model)
    (persist_dir / METADATA_FILE).write_text(json.dumps({"embedding_model": current_model_name}), encoding="utf-8")
    return index


# --- Initialize embedding model ---
embed_model = HuggingFaceEmbedding(
    model_name = EMBEDDING_MODEL_NAME,
    device     = "cpu",
)

# --- Load and index document ---
pdf_path = UPLOADS_DIR / "openstax_ch21_carboxylic_acid_derivatives.pdf"

doc = IngestedDocument(
    input_type  = InputType.PDF,
    text        = extract_text_from_pdf(pdf_path),
    source_path = pdf_path,
)

chunks = fixed_size_chunking(doc)
index  = load_and_verify_index(
    persist_dir        = INDEX_DIR / "ch21",
    collection_name    = "openstax_ch21",
    current_model_name = EMBEDDING_MODEL_NAME,
    embed_model        = embed_model,
    chunks             = chunks,
)

# --- Demo queries ---
queries = ["amide", "ester"]

for query in queries:
    print(f"\n{'='*50}")
    print(f"Query: {query}")
    print('='*50)

    print("\n--- Flash Card ---")
    card = rag_query_with_validation(
        query       = query,
        index       = index,
        llm         = llm,
        embed_model = embed_model,
        mode        = "flashcard",
    )
    print(f"Concept    : {card.concept}")
    print(f"Definition : {card.definition}")
    print(f"Key points : {card.key_points}")
    print(f"Examples   : {card.examples}")
    print(f"Image      : {card.suggested_image}")

    print("\n--- Consolidated Summary ---")
    summary = rag_query_with_validation(
        query       = query,
        index       = index,
        llm         = llm,
        embed_model = embed_model,
        mode        = "summary",
    )
    print(f"Topic    : {summary.topic}")
    print(f"Overview : {summary.overview}")
    for c in summary.concepts:
        print(f"  - {c['name']}: {c['description']}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Query: amide

--- Flash Card ---
Concept    : amide reduction
Definition : Amide reduction involves the reduction of amides with LiAlH4 to form amines. The reaction proceeds through nucleophilic addition of hydride ion to the amide carbonyl group, followed by expulsion of the oxygen atom as an aluminate anion leaving group to give an iminium ion intermediate. The iminium ion is further reduced by LiAlH4 to yield the amine.
Key points : ['reduction with LiAlH4', 'nucleophilic addition of hydride ion', 'expulsion of oxygen atom as aluminate anion', 'further reduction by LiAlH4']
Examples   : ['N-ethylaniline from N-phenylacetamide', 'N-ethylbenzamide to benzoic acid, benzyl alcohol, and N,N-dimethylaminomethylcyclohexane']
Image      : A chemical reaction showing the reduction of an amide with LiAlH4, resulting in the formation of an amine

--- Consolidated Summary ---
Schema violation on attempt 1: Unterminated string starting at: line 23 column 28 (char 1686)
Retrying with example-aug